# Privacy, PII and Data Governance Analysis
## NovaCred Credit Application Dataset - Governance Officer 

#### We will, in the following notebook, deeply analyse the potential risks for privacy and gorvernance of the NovaCred credit application dataset. By setting ourselves different objectives such as pin pointing personal identifiable informations, map findings to GDPR requirements and more, we will aim to propose governance improvements for the NovaCard system. 

In [13]:
import pandas as pd
import json
from pathlib import Path
from IPython.display import display, HTML

# Define file path
data_path = Path("../Data/raw_credit_applications.json")

# Load raw JSON file
with open(data_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Convert to DataFrame
df = pd.DataFrame(raw_data)

# Basic preview
print(f"Number of records: {len(df)}")
print(f"Columns: {list(df.columns)}")

df.head()

Number of records: 502
Columns: ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp', 'loan_purpose', 'notes']


,_id,applicant_info,financials,spending_behavior,decision,processing_timestamp,loan_purpose,notes
0,app_200,"{'full_name': 'Jerry Smith', 'email': 'jerry.s...","{'annual_income': 73000, 'credit_history_month...","[{'category': 'Shopping', 'amount': 480}, {'ca...","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,"{'full_name': 'Brandon Walker', 'email': 'bran...","{'annual_income': 78000, 'credit_history_month...","[{'category': 'Rent', 'amount': 608}, {'catego...","{'loan_approved': False, 'rejection_reason': '...",NaN,NaN,NaN
2,app_215,"{'full_name': 'Scott Moore', 'email': 'scott.m...","{'annual_income': 61000, 'credit_history_month...","[{'category': 'Rent', 'amount': 109}]","{'loan_approved': True, 'interest_rate': 3.7, ...",NaN,vacation,NaN
3,app_024,"{'full_name': 'Thomas Lee', 'email': 'thomas.l...","{'annual_income': 103000, 'credit_history_mont...","[{'category': 'Fitness', 'amount': 575}]","{'loan_approved': True, 'interest_rate': 4.3, ...",NaN,NaN,NaN
4,app_184,"{'full_name': 'Brian Rodriguez', 'email': 'bri...","{'annual_income': 57000, 'credit_history_month...","[{'category': 'Entertainment', 'amount': 463}]","{'loan_approved': False, 'rejection_reason': '...",2024-01-15T00:00:00Z,NaN,NaN


#### We flatten the data set, as it contains a nested structure, to use it in a more effective way accross the rest of our analyses. 

In [7]:
applicant_df = pd.json_normalize(df["applicant_info"]).add_prefix("applicant_")
financials_df = pd.json_normalize(df["financials"]).add_prefix("financial_")
decision_df = pd.json_normalize(df["decision"]).add_prefix("decision_")

# Combine into a single dataframe
df_flat = pd.concat(
    [
        df["_id"],
        applicant_df,
        financials_df,
        decision_df,
        df["spending_behavior"],
        df["processing_timestamp"],
        df["loan_purpose"],
        df["notes"],
    ],
    axis=1,
)

# Inspect result
df_flat.head()

,_id,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_gender,applicant_date_of_birth,applicant_zip_code,financial_annual_income,financial_credit_history_months,...,financial_savings_balance,financial_annual_salary,decision_loan_approved,decision_rejection_reason,decision_interest_rate,decision_approved_amount,spending_behavior,processing_timestamp,loan_purpose,notes
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,...,31212,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,NaN,NaN
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,...,17915,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,NaN,NaN
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,...,37909,NaN,True,NaN,3.7,59000.0,"[{'category': 'Rent', 'amount': 109}]",NaN,vacation,NaN
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,70,...,0,NaN,True,NaN,4.3,34000.0,"[{'category': 'Fitness', 'amount': 575}]",NaN,NaN,NaN
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,14,...,31763,NaN,False,algorithm_risk_score,NaN,NaN,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,NaN,NaN


In [9]:
# Visualization setup for governance reporting
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Simple governance-oriented color palette
GOV_COLORS = {
    "critical": "#B03A2E",   # dark red
    "high": "#D35400",       # orange
    "medium": "#CA8A04",     # amber
    "low": "#2E86AB",        # blue
    "ok": "#2E8B57",         # green
    "text": "#2C3E50",       # dark gray-blue
    "grid": "#E5E7EB",       # light gray
    "bg": "#FAFAFA"          # soft background
}

# Global plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.facecolor": GOV_COLORS["bg"],
    "axes.facecolor": "white",
    "axes.edgecolor": "#D0D7DE",
    "axes.labelcolor": GOV_COLORS["text"],
    "axes.titlecolor": GOV_COLORS["text"],
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "xtick.color": GOV_COLORS["text"],
    "ytick.color": GOV_COLORS["text"],
    "grid.color": GOV_COLORS["grid"],
    "grid.linestyle": "--",
    "grid.linewidth": 0.7,
    "font.size": 11,
    "legend.frameon": False
})

print("Governance visualization style loaded.")

Governance visualization style loaded.


# 1 - PII Identification 

In [12]:
pii_sample = df_flat[
    [
        "applicant_full_name",
        "applicant_email",
        "applicant_ssn",
        "applicant_ip_address",
        "applicant_date_of_birth",
        "applicant_gender",
        "applicant_zip_code"
    ]
].head()

pii_sample

,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_date_of_birth,applicant_gender,applicant_zip_code
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,2001-03-09,Male,10036
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,1992-03-31,M,10032
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,1989-10-24,Male,10075
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,1983-04-25,Male,10077
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,1999-05-21,M,10080


In [15]:
# --- PII inventory with GDPR mapping ---

def highlight_risk(val):
    colors = {
        "Critical": "background-color:#F8D7DA; color:#721C24; font-weight:bold;",
        "High": "background-color:#FDE2C8; color:#7C2D12;",
        "Medium": "background-color:#FFF3CD; color:#664D03;",
        "Low": "background-color:#D1E7DD; color:#0F5132;"
    }
    return colors.get(val, "")

pii_inventory = pd.DataFrame({
    "field_name": [
        "applicant_full_name",
        "applicant_email",
        "applicant_ssn",
        "applicant_ip_address",
        "applicant_date_of_birth",
        "applicant_gender",
        "applicant_zip_code"
    ],
    "pii_category": [
        "Direct identifier",
        "Direct identifier",
        "Sensitive identifier",
        "Online identifier",
        "Personal / demographic data",
        "Protected attribute",
        "Quasi-identifier"
    ],
    "risk_level": [
        "High",
        "High",
        "Critical",
        "High",
        "Medium",
        "High",
        "Medium"
    ],
    "gdpr_article_or_principle": [
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(c), 5(1)(f) - Minimization & confidentiality",
        "Art. 4(1), 5(1)(f) - Personal data & confidentiality",
        "Art. 5(1)(c) - Data minimization",
        "Art. 5(1)(a), Art. 22 - Fairness & automated decision-making",
        "Art. 5(1)(c), Art. 22 - Minimization & discrimination risk"
    ],
    "governance_risk": [
        "Directly identifies the applicant",
        "Directly identifies the applicant and exposes contact information",
        "Highly sensitive identifier that should not be stored in plain text",
        "Can be linked to an individual or device context",
        "Can support re-identification when combined with other fields",
        "May create fairness and discrimination risk in lending decisions",
        "May act as a proxy for socioeconomic status or ethnicity"
    ]
})

pii_inventory.style.applymap(
    highlight_risk,
    subset=["risk_level"]
).set_properties(**{
    "text-align": "left"
})

/var/folders/6x/48zsnbjs4hz2kb565wlpx3800000gn/T/ipykernel_17573/3561871187.py:60: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  pii_inventory.style.applymap(


,field_name,pii_category,risk_level,gdpr_article_or_principle,governance_risk
0,applicant_full_name,Direct identifier,High,Art. 5(1)(c) - Data minimization,Directly identifies the applicant
1,applicant_email,Direct identifier,High,Art. 5(1)(c) - Data minimization,Directly identifies the applicant and exposes contact information
2,applicant_ssn,Sensitive identifier,Critical,"Art. 5(1)(c), 5(1)(f) - Minimization & confidentiality",Highly sensitive identifier that should not be stored in plain text
3,applicant_ip_address,Online identifier,High,"Art. 4(1), 5(1)(f) - Personal data & confidentiality",Can be linked to an individual or device context
4,applicant_date_of_birth,Personal / demographic data,Medium,Art. 5(1)(c) - Data minimization,Can support re-identification when combined with other fields
5,applicant_gender,Protected attribute,High,"Art. 5(1)(a), Art. 22 - Fairness & automated decision-making",May create fairness and discrimination risk in lending decisions
6,applicant_zip_code,Quasi-identifier,Medium,"Art. 5(1)(c), Art. 22 - Minimization & discrimination risk",May act as a proxy for socioeconomic status or ethnicity


#### As some of those informations are set between medium and critical risk level, we want to analyse to what extent those PII fileds are exposed and present in our data set. 

In [16]:
# Basic completeness and exposure analysis for key PII fields

pii_cols = [
    "applicant_full_name",
    "applicant_email",
    "applicant_ssn",
    "applicant_ip_address",
    "applicant_date_of_birth",
    "applicant_gender",
    "applicant_zip_code"
]

pii_summary = pd.DataFrame({
    "field_name": pii_cols,
    "non_null_count": [df_flat[col].notna().sum() for col in pii_cols],
    "missing_count": [df_flat[col].isna().sum() for col in pii_cols],
    "missing_pct": [round(df_flat[col].isna().mean() * 100, 2) for col in pii_cols],
    "unique_values": [df_flat[col].nunique(dropna=True) for col in pii_cols]
})

pii_summary

,field_name,non_null_count,missing_count,missing_pct,unique_values
0,applicant_full_name,502,0,0.0,475
1,applicant_email,502,0,0.0,494
2,applicant_ssn,497,5,1.0,494
3,applicant_ip_address,497,5,1.0,496
4,applicant_date_of_birth,501,1,0.2,494
5,applicant_gender,501,1,0.2,5
6,applicant_zip_code,501,1,0.2,196


#### We see that the PII filed create privacy, compliance and fairness risks for NovaCred. We have both direct and indirect identifiers in our categories. 
- **Direct identifiers** such as full name, email, and SSN can immediately identify an applicant.
- **Online identifiers** such as IP address are personal data under GDPR and may reveal device or location context.
- **Quasi-identifiers** such as date of birth and ZIP code may not identify someone alone, but can enable re-identification when combined with other fields.
- **Protected attributes** such as gender create additional fairness and discrimination risks in automated credit decisions.

#### We need to execute a brief deep dive into the SSN, as the sensitivity of it can lead to serious consequences such as financial fraud, or impersonation from outsiders if the data is leaked. We could assume that this violates GDPR data minimization as a machine learning model should not rquire SSN to predict credit worthiness (if not proved otherwise) 

# 2 - GDPR Article Mapping 

In [22]:
# --- GDPR mapping and governance risk analysis (readable display) ---

from IPython.display import display, HTML

# Make long text fully visible in Jupyter
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

gdpr_mapping = pd.DataFrame({
    "dataset_issue": [
        "Raw identifiers stored (name, email)",
        "SSN stored in plain text",
        "IP address collected and stored",
        "Demographic attributes present (DOB, gender)",
        "No consent tracking visible",
        "No retention or deletion metadata",
        "Automated credit decisions recorded"
    ],
    
    "relevant_gdpr_article": [
        "Art. 5(1)(c) – Data Minimization",
        "Art. 5(1)(f) – Integrity & Confidentiality",
        "Art. 4(1) – Definition of Personal Data",
        "Art. 5(1)(a), Art. 22 – Fairness & Automated Decisions",
        "Art. 6 – Lawfulness of Processing",
        "Art. 5(1)(e), Art. 17 – Storage Limitation & Right to Erasure",
        "Art. 22 – Automated Decision-Making"
    ],
    
    "why_dangerous": [
        "Direct identifiers allow immediate identification of individuals.",
        "SSNs uniquely identify individuals and are highly sensitive national identifiers.",
        "IP addresses may reveal device, location, or behavioral patterns.",
        "Demographic attributes may introduce discrimination risks in credit decisions.",
        "Absence of consent tracking makes the legal basis of processing unclear.",
        "Lack of retention metadata may lead to indefinite storage of personal data.",
        "Automated credit decisions may affect individuals without meaningful human review."
    ],
    
    "juridical_consequences": [
        "Violation of GDPR data minimization principles and possible regulatory investigation.",
        "Severe GDPR penalties for insufficient protection of sensitive personal identifiers.",
        "Legal classification as personal data requiring appropriate security safeguards.",
        "Potential discrimination claims and regulatory scrutiny of algorithmic fairness.",
        "Unlawful processing under GDPR if no valid legal basis is documented.",
        "Non-compliance with storage limitation and right-to-erasure obligations.",
        "Violation of Article 22 protections regarding automated decision-making."
    ],
    
    "governance_obligation": [
        "Limit collection and exposure of direct identifiers.",
        "Apply pseudonymization or strong encryption controls.",
        "Protect online identifiers and restrict access.",
        "Monitor fairness and prevent discriminatory outcomes.",
        "Document legal basis and consent mechanisms.",
        "Define retention policies and deletion mechanisms.",
        "Implement human oversight for automated decisions."
    ]
})

# Display as styled HTML table with wrapped text
display(
    HTML(
        gdpr_mapping.to_html(index=False).replace(
            '<table border="1" class="dataframe">',
            '<table border="1" class="dataframe" style="width:100%; table-layout:fixed; border-collapse:collapse;">'
        )
    )
)

# Apply notebook-level CSS for readability
display(HTML("""
<style>
table.dataframe td, table.dataframe th {
    white-space: normal !important;
    word-wrap: break-word;
    text-align: left !important;
    vertical-align: top;
    padding: 8px;
}
table.dataframe th {
    background-color: #EAF2F8;
    font-weight: bold;
}
</style>
"""))

dataset_issue,relevant_gdpr_article,why_dangerous,juridical_consequences,governance_obligation
"Raw identifiers stored (name, email)",Art. 5(1)(c) – Data Minimization,Direct identifiers allow immediate identification of individuals.,Violation of GDPR data minimization principles and possible regulatory investigation.,Limit collection and exposure of direct identifiers.
SSN stored in plain text,Art. 5(1)(f) – Integrity & Confidentiality,SSNs uniquely identify individuals and are highly sensitive national identifiers.,Severe GDPR penalties for insufficient protection of sensitive personal identifiers.,Apply pseudonymization or strong encryption controls.
IP address collected and stored,Art. 4(1) – Definition of Personal Data,"IP addresses may reveal device, location, or behavioral patterns.",Legal classification as personal data requiring appropriate security safeguards.,Protect online identifiers and restrict access.
"Demographic attributes present (DOB, gender)","Art. 5(1)(a), Art. 22 – Fairness & Automated Decisions",Demographic attributes may introduce discrimination risks in credit decisions.,Potential discrimination claims and regulatory scrutiny of algorithmic fairness.,Monitor fairness and prevent discriminatory outcomes.
No consent tracking visible,Art. 6 – Lawfulness of Processing,Absence of consent tracking makes the legal basis of processing unclear.,Unlawful processing under GDPR if no valid legal basis is documented.,Document legal basis and consent mechanisms.
No retention or deletion metadata,"Art. 5(1)(e), Art. 17 – Storage Limitation & Right to Erasure",Lack of retention metadata may lead to indefinite storage of personal data.,Non-compliance with storage limitation and right-to-erasure obligations.,Define retention policies and deletion mechanisms.
Automated credit decisions recorded,Art. 22 – Automated Decision-Making,Automated credit decisions may affect individuals without meaningful human review.,Violation of Article 22 protections regarding automated decision-making.,Implement human oversight for automated decisions.


#### Under GDPR Article 22, individuals have protections when decisions are made solely by automated means and those decisions produce legal or similarly significant effects, such as credit approval or rejection.
To assess this risk, we inspect the decision-related fields in the dataset and look for evidence of:
- automated approval / rejection outputs
- algorithmic rejection reasons
- absence of human oversight indicators

In [23]:
# Inspect the decision-related fields
decision_cols = [
    "decision_loan_approved",
    "decision_interest_rate",
    "decision_approved_amount",
    "decision_rejection_reason"
]

df_flat[decision_cols].head(10)

,decision_loan_approved,decision_interest_rate,decision_approved_amount,decision_rejection_reason
0,False,NaN,NaN,algorithm_risk_score
1,False,NaN,NaN,algorithm_risk_score
2,True,3.7,59000.0,NaN
3,True,4.3,34000.0,NaN
4,False,NaN,NaN,algorithm_risk_score
5,False,NaN,NaN,algorithm_risk_score
6,True,5.6,27000.0,NaN
7,True,2.8,38000.0,NaN
8,False,NaN,NaN,algorithm_risk_score
9,False,NaN,NaN,insufficient_credit_history


In [28]:
from IPython.display import display, HTML

# --- Approvals vs rejections ---
decision_summary = (
    df_flat["decision_loan_approved"]
    .value_counts(dropna=False)
    .rename_axis("loan_approved")
    .reset_index(name="count")
)

# --- Rejection reasons ---
rejection_reason_counts = (
    df_flat["decision_rejection_reason"]
    .fillna("No rejection reason")
    .value_counts()
    .reset_index()
)

rejection_reason_counts.columns = ["rejection_reason", "count"]

# --- Potentially automated rejection reasons ---
automated_keywords = ["algorithm", "risk_score", "model", "score", "automated"]

auto_rejections = df_flat[
    df_flat["decision_rejection_reason"]
    .fillna("")
    .str.lower()
    .str.contains("|".join(automated_keywords), regex=True)
]

auto_rejection_sample = auto_rejections[[
    "_id",
    "decision_loan_approved",
    "decision_rejection_reason"
]].head(10)

# --- Display side-by-side ---
display(
    HTML(
        f"""
        <div style="display:flex; gap:40px; align-items:flex-start">

            <div style="width:30%">
                <h4>Approval vs Rejection</h4>
                {decision_summary.to_html(index=False)}
            </div>

            <div style="width:35%">
                <h4>Rejection Reasons</h4>
                {rejection_reason_counts.to_html(index=False)}
            </div>

            <div style="width:35%">
                <h4>Potential Automated Rejections</h4>
                {auto_rejection_sample.to_html(index=False)}
            </div>

        </div>
        """
    )
)

The dataset provides clear evidence that NovaCred stores automated credit decision outputs, including approval status and rejection reasons.

If rejection reasons include terms such as `algorithm_risk_score`, this suggests that some applicants are denied credit through an algorithmic decision rule rather than a documented human review process.

From a governance perspective, this raises a potential **GDPR Article 22 risk**, since applicants may be subject to decisions with significant legal or financial effects without visible evidence of human intervention, contestability, or explanation.

## 3. Pseudonymization & Anonymization

Under the GDPR, organizations must implement technical and organizational measures to reduce the risk associated with storing personal data.

**GDPR Article 4(5)** defines *pseudonymization* as the processing of personal data in a way that prevents direct attribution to a specific individual without additional information kept separately.

**GDPR Article 32** further recommends pseudonymization and encryption as appropriate safeguards to ensure the security and confidentiality of personal data.

In the NovaCred dataset, several fields contain direct identifiers or sensitive attributes. To reduce privacy risks while preserving analytical usefulness, multiple privacy-preserving techniques can be applied to these fields.

The table below outlines the privacy protection strategies considered for different categories of personal data present in the dataset.

In [33]:
# --- Pseudonymization of sensitive identifiers ---

import hashlib

def pseudonymize(value):
    """
    Convert a personal identifier into a pseudonymized hash using SHA-256.
    """
    if pd.isna(value):
        return None
    
    value_str = str(value)
    hashed_value = hashlib.sha256(value_str.encode()).hexdigest()[:16]  # shortened for readability
    
    return hashed_value


# Create pseudonymized columns
df_flat["applicant_name_pseudo"] = df_flat["applicant_full_name"].apply(pseudonymize)

df_flat["applicant_email_pseudo"] = df_flat["applicant_email"].apply(pseudonymize)

df_flat["applicant_ssn_pseudo"] = df_flat["applicant_ssn"].apply(pseudonymize)


# Display comparison of original vs pseudonymized identifiers
df_flat[
    [
        "applicant_full_name",
        "applicant_name_pseudo",
        "applicant_email",
        "applicant_email_pseudo",
        "applicant_ssn",
        "applicant_ssn_pseudo"
    ]
].head(10)

,applicant_full_name,applicant_name_pseudo,applicant_email,applicant_email_pseudo,applicant_ssn,applicant_ssn_pseudo
0,Jerry Smith,68ee17cf46b05603,jerry.smith17@hotmail.com,116648a776152574,596-64-4340,2caf30528c21a10e
1,Brandon Walker,4c539f3c4c8794d5,brandon.walker2@yahoo.com,c3522c0b54ef9045,425-69-4784,2f7da45fefdcfb2c
2,Scott Moore,4ad1a6eb65ea2135,scott.moore94@mail.com,b299e7d6a37e183b,370-78-5178,db120edcee2366a4
3,Thomas Lee,0b6a308e6a3e28e0,thomas.lee6@protonmail.com,6fbd2478748a29fa,194-35-1833,c835719be0201898
4,Brian Rodriguez,099b981258dca619,brian.rodriguez86@aol.com,f24e7cc1450ee9aa,480-41-2475,41c7de40dc491858
5,Maria Miller,e000c1c9876a7cda,maria.miller67@outlook.com,8843b2c0ec23996a,417-25-4912,a0a5f9cec1ea52aa
6,Nicholas King,52f245abedecbea4,nicholas.king46@outlook.com,03c1a6e2b0058c37,613-23-2503,51280b2e354d80c6
7,Susan Rivera,de7677e8bcad5409,susan.rivera74@gmail.com,67c6d13632c3404f,176-97-1864,04adc2ff16245195
8,Joseph Lopez,750c74fb22b712d1,joseph.lopez1@gmail.com,a7f7177878859512,652-70-5530,fb789d550e179a5c
9,Michael Mitchell,dcb77bc29d159f43,michael.mitchell42@hotmail.com,f6a2496402be6666,100-94-8400,0befb0709774eeb4


The table above demonstrates pseudonymization applied to the most sensitive direct identifiers in the dataset: applicant name, email, and SSN.

Each value is transformed using SHA-256 hashing, producing a consistent pseudonymous identifier that prevents direct identification while preserving the ability to link records internally.

This approach supports GDPR principles of **data minimization (Art. 5(1)(c))** and **security of processing (Art. 32)** by reducing the exposure of raw identifiers in analytical environments.

## 4. Right to Erasure Simulation

Under **GDPR Article 17**, data subjects have the right to request the deletion of their personal data under certain conditions.

In practice, this means NovaCred should be able to identify a specific applicant record and remove or anonymize personal identifiers in a controlled and documented way.

The following simulation illustrates how a right-to-erasure request could be handled at the dataset level.

In [36]:
# Select one example applicant record
erasure_demo = df_flat[[
    "_id",
    "applicant_full_name",
    "applicant_email",
    "applicant_ssn",
    "applicant_ip_address",
    "applicant_date_of_birth",
    "applicant_gender",
    "applicant_zip_code"
]].head(1).copy()

# Simulate erasure
erased_record = erasure_demo.copy()

fields_to_erase = [
    "applicant_full_name",
    "applicant_email",
    "applicant_ssn",
    "applicant_ip_address",
    "applicant_date_of_birth",
    "applicant_zip_code"
]

for col in fields_to_erase:
    erased_record[col] = "[ERASED]"

In [38]:
from IPython.display import display, HTML

display(
    HTML(
        f"""
        <div style="display:flex; flex-direction:column; gap:30px">

            <div>
                <h4>Original Record</h4>
                {erasure_demo.to_html(index=False)}
            </div>

            <div>
                <h4>After Erasure Simulation</h4>
                {erased_record.to_html(index=False)}
            </div>

        </div>
        """
    )
)

_id,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_date_of_birth,applicant_gender,applicant_zip_code
app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,2001-03-09,Male,10036
_id,applicant_full_name,applicant_email,applicant_ssn,applicant_ip_address,applicant_date_of_birth,applicant_gender,applicant_zip_code
app_200,[ERASED],[ERASED],[ERASED],[ERASED],[ERASED],Male,[ERASED]


This simulation shows how direct identifiers and quasi-identifiers could be removed from a record in response to a deletion request.

From a governance perspective, the right to erasure requires more than manual deletion in a notebook. NovaCred would need a documented operational process that includes:

- identification of the data subject
- traceability across systems and storage layers
- deletion or anonymization of relevant fields
- retention exceptions where legal obligations require temporary preservation

The current dataset does not contain visible metadata such as deletion status, retention expiry, or erasure logs, which limits auditability of this right in practice.

## 5. EU AI Act Classification

Under the EU AI Act, certain AI systems are classified as **High-Risk** when they significantly affect individuals' access to essential services.

Credit scoring and creditworthiness assessments fall into this category because they influence a person's ability to obtain financial services.

The NovaCred system processes applicant financial and behavioral data to determine whether a loan should be approved or rejected. As a result, the system qualifies as a **High-Risk AI System** under the EU AI Act.

High-risk AI systems must comply with strict requirements related to risk management, transparency, data governance, and human oversight.

In [39]:
ai_act_classification = pd.DataFrame({
    "system_component": [
        "Credit scoring algorithm",
        "Automated loan approval decision",
        "Use of behavioral financial data",
        "Use of demographic attributes"
    ],
    "ai_act_risk_level": [
        "High-Risk",
        "High-Risk",
        "High-Risk",
        "Potential fairness risk"
    ],
    "regulatory_reference": [
        "AI Act Annex III – Creditworthiness assessment",
        "AI Act Article 6 – High-risk systems",
        "AI Act Annex III",
        "AI Act Article 10 – Data governance"
    ],
    "governance_requirement": [
        "Risk management framework",
        "Human oversight and explainability",
        "Data quality and bias monitoring",
        "Fairness monitoring and documentation"
    ]
})

ai_act_classification

,system_component,ai_act_risk_level,regulatory_reference,governance_requirement
0,Credit scoring algorithm,High-Risk,AI Act Annex III – Creditworthiness assessment,Risk management framework
1,Automated loan approval decision,High-Risk,AI Act Article 6 – High-risk systems,Human oversight and explainability
2,Use of behavioral financial data,High-Risk,AI Act Annex III,Data quality and bias monitoring
3,Use of demographic attributes,Potential fairness risk,AI Act Article 10 – Data governance,Fairness monitoring and documentation


The analysis confirms that NovaCred's credit decision system falls within the **High-Risk AI category** defined by the EU AI Act.

This classification introduces several governance obligations, including:

- implementation of a formal **risk management system**
- **documentation of model logic and training data**
- **human oversight mechanisms** for automated decisions
- monitoring of **bias and discriminatory outcomes**
- maintaining **audit logs and traceability**

These requirements complement existing GDPR obligations, particularly those related to automated decision-making under **Article 22**.

In [40]:
ai_governance_gap = pd.DataFrame({
    "ai_act_requirement": [
        "Risk management system",
        "Training data governance",
        "Human oversight",
        "Transparency to applicants",
        "Audit logging"
    ],
    "visible_in_dataset": [
        "No",
        "Partially",
        "No",
        "No",
        "No"
    ],
    "risk_implication": [
        "Model risks not formally assessed",
        "Potential bias from demographic attributes",
        "Automated decisions without manual review",
        "Applicants may not understand decision logic",
        "Limited traceability of decisions"
    ]
})

ai_governance_gap

,ai_act_requirement,visible_in_dataset,risk_implication
0,Risk management system,No,Model risks not formally assessed
1,Training data governance,Partially,Potential bias from demographic attributes
2,Human oversight,No,Automated decisions without manual review
3,Transparency to applicants,No,Applicants may not understand decision logic
4,Audit logging,No,Limited traceability of decisions


## 6. Governance & Oversight Controls

The previous analysis identified several privacy, fairness, and regulatory risks in the NovaCred dataset and decision process.

To address these risks, NovaCred should implement a set of governance and oversight controls aligned with both **GDPR requirements** and the **EU AI Act obligations for high-risk systems**.

The table below summarizes recommended governance controls and their expected impact on regulatory compliance.

In [41]:
governance_controls = pd.DataFrame({
    "risk_area": [
        "Personal data exposure",
        "Sensitive identifiers",
        "Automated credit decisions",
        "Bias and discrimination risk",
        "Data retention",
        "Consent and lawful basis",
        "Model transparency"
    ],
    
    "identified_issue": [
        "Direct identifiers stored in dataset",
        "SSN and email visible in raw form",
        "Loan decisions appear fully automated",
        "Demographic attributes present",
        "No retention or deletion metadata",
        "Consent status not recorded",
        "Decision logic not documented"
    ],
    
    "recommended_control": [
        "Apply pseudonymization before analytics access",
        "Encrypt identifiers and restrict access",
        "Implement human review for rejected applications",
        "Monitor fairness metrics and disparate impact",
        "Define retention and deletion policies",
        "Record lawful basis for data processing",
        "Provide explainability for credit decisions"
    ],
    
    "regulatory_basis": [
        "GDPR Art.5(1)(c), Art.32",
        "GDPR Art.32",
        "GDPR Art.22",
        "GDPR Art.5(1)(a), AI Act Art.10",
        "GDPR Art.5(1)(e), Art.17",
        "GDPR Art.6",
        "AI Act transparency requirements"
    ]
})

governance_controls

,risk_area,identified_issue,recommended_control,regulatory_basis
0,Personal data exposure,Direct identifiers stored in dataset,Apply pseudonymization before analytics access,"GDPR Art.5(1)(c), Art.32"
1,Sensitive identifiers,SSN and email visible in raw form,Encrypt identifiers and restrict access,GDPR Art.32
2,Automated credit decisions,Loan decisions appear fully automated,Implement human review for rejected applications,GDPR Art.22
3,Bias and discrimination risk,Demographic attributes present,Monitor fairness metrics and disparate impact,"GDPR Art.5(1)(a), AI Act Art.10"
4,Data retention,No retention or deletion metadata,Define retention and deletion policies,"GDPR Art.5(1)(e), Art.17"
5,Consent and lawful basis,Consent status not recorded,Record lawful basis for data processing,GDPR Art.6
6,Model transparency,Decision logic not documented,Provide explainability for credit decisions,AI Act transparency requirements
